# 7.7 — Vocabulary Size and Document Frequency

**Chapter 7, section 7.5.2.**

**The question this notebook answers:** what do `vocabSize`, `minDF` and `maxDF` actually change?

They are constructor arguments to `CountVectorizer`, and it is easy to read them as tidying. They
are not. Between them they decide **which terms have a dimension at all**, and therefore what
every coordinate of every document vector means. The same pipeline is fitted three times over the
same corpus with three settings, and the three feature spaces are compared term by term.

Source: `Notebooks old/Spark-Example-24-Mlib-LDA.ipynb`, cells 10–14 and 23–25, rebuilt. The
original's three settings were commented-out alternatives, run one at a time by editing the cell;
here they run together so they can be compared. The notebook's LDA half is topic modelling, which
is not in this chapter's scope, and is not carried over.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.7")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The same corpus as notebook 7.6

A quarter of the film reviews, chosen by hashing the review text so that the counts are the same
on every machine. The cleaning is 7.6's, plus the whitespace fix of its section 5, so that the
empty string does not occupy dimension 0 and distract from what is being compared here.

In [2]:
import pyspark.sql.functions as F

reviews = spark.read.csv(os.path.join(DATA, "IMDB Dataset.csv.bz2"), header=True)

cleaned = F.regexp_replace(F.col("text"), r'<[^>]+>', "")
cleaned = F.regexp_replace(cleaned, r"[^a-zA-Z ]", "")
cleaned = F.regexp_replace(cleaned, r"\b\w{1,2}\b", "")
cleaned = F.trim(F.regexp_replace(cleaned, r"\s+", " "))       # the fix from 7.6 section 5

docs = (reviews
        .withColumn("bucket", F.crc32(F.col("text")) % 100)
        .filter("bucket < 25")
        .withColumn("text_c", F.lower(cleaned))
        .select("text_c")
        .cache())

n_docs = docs.count()
print("documents:", n_docs)

documents: 12520


---

## 2. One pipeline, fitted three ways

The stages never change. Only the three arguments do.

* **A** — `vocabSize=500`. The five hundred commonest terms.
* **B** — `vocabSize=1000`. Twice as many.
* **C** — `vocabSize=1000, minDF=10, maxDF=5000`. Still at most a thousand terms, but a term is
  now eligible only if it appears in **at least ten** documents and **at most five thousand**.
* **D** — the same, with `maxDF=2500`. A fifth of the corpus rather than two-fifths, to show what
  the threshold does when it actually bites.

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF


def fit_vectorizer(**cv_args):
    tokenizer = Tokenizer(inputCol="text_c", outputCol="words")
    remover = StopWordsRemover(inputCol="words", outputCol="filtered")
    countVectorizer = CountVectorizer(inputCol="filtered", outputCol="rawFeatures", **cv_args)
    idf = IDF(inputCol="rawFeatures", outputCol="featuresIDF")
    return Pipeline(stages=[tokenizer, remover, countVectorizer, idf]).fit(docs)


runs = {
    "A  vocabSize=500":                        fit_vectorizer(vocabSize=500),
    "B  vocabSize=1000":                       fit_vectorizer(vocabSize=1000),
    "C  vocabSize=1000, minDF=10, maxDF=5000": fit_vectorizer(vocabSize=1000,
                                                              minDF=10.0, maxDF=5000.0),
    "D  vocabSize=1000, minDF=10, maxDF=2500": fit_vectorizer(vocabSize=1000,
                                                              minDF=10.0, maxDF=2500.0),
}

for name, model in runs.items():
    print(f"{name:<42} {len(model.stages[2].vocabulary):>5} terms")

A  vocabSize=500                             500 terms
B  vocabSize=1000                           1000 terms
C  vocabSize=1000, minDF=10, maxDF=5000     1000 terms
D  vocabSize=1000, minDF=10, maxDF=2500     1000 terms


`vocabSize` is a **ceiling, not a target**: the document-frequency thresholds decide which terms
are eligible, and the ranking by frequency then fills the ceiling from what is left. All four runs
here happen to fill it, because this corpus has far more than a thousand eligible terms — but the
thousand terms are not the same thousand, and section 6 shows settings under which the ceiling is
not reached at all.

In [4]:
vocabs = {name: model.stages[2].vocabulary for name, model in runs.items()}

for name, vocab in vocabs.items():
    print(f"{name}")
    print(f"   first 12:  {vocab[:12]}")
    print(f"   last  6:   {vocab[-6:]}")
    print()

A  vocabSize=500
   first 12:  ['movie', 'film', 'one', 'like', 'good', 'even', 'time', 'really', 'see', 'story', 'much', 'well']
   last  6:   ['whose', 'jokes', 'number', 'hilarious', 'god', 'knew']

B  vocabSize=1000
   first 12:  ['movie', 'film', 'one', 'like', 'good', 'even', 'time', 'really', 'see', 'story', 'much', 'well']
   last  6:   ['blue', 'journey', 'nudity', 'recommended', 'tough', 'walk']

C  vocabSize=1000, minDF=10, maxDF=5000
   first 12:  ['good', 'even', 'time', 'really', 'see', 'story', 'much', 'well', 'get', 'people', 'great', 'bad']
   last  6:   ['tough', 'walk', 'frank', 'language', 'credit', 'bought']

D  vocabSize=1000, minDF=10, maxDF=2500
   first 12:  ['films', 'never', 'many', 'two', 'love', 'character', 'best', 'know', 'little', 'plot', 'life', 'show']
   last  6:   ['respect', 'stick', 'amount', 'appearance', 'ghost', 'likes']



---

## 3. What the thresholds removed and what they let in

`maxDF=5000` excludes any term appearing in more than five thousand of the documents. On a corpus
of this size that is the top of the frequency list — exactly the terms a stop-word list would have
removed if they had been on it. `minDF=10` excludes terms appearing in fewer than ten documents,
which is where misspellings, proper nouns and one-off coinages live.

In [5]:
a, b, c, d = (vocabs["A  vocabSize=500"],
              vocabs["B  vocabSize=1000"],
              vocabs["C  vocabSize=1000, minDF=10, maxDF=5000"],
              vocabs["D  vocabSize=1000, minDF=10, maxDF=2500"])

for label, other in (("C  (maxDF=5000)", c), ("D  (maxDF=2500)", d)):
    dropped = [t for t in b if t not in set(other)]
    added = [t for t in other if t not in set(b)]
    print(f"{label}  vs  B")
    print(f"   {len(dropped):>3} terms of B are gone. The most frequent of them:")
    print(f"       {dropped[:14]}")
    print(f"   {len(added):>3} terms take their place. The first of them:")
    print(f"       {added[:14]}")
    print()

dropped = [t for t in b if t not in set(d)]

C  (maxDF=5000)  vs  B
     4 terms of B are gone. The most frequent of them:
       ['movie', 'film', 'one', 'like']
     4 terms take their place. The first of them:
       ['frank', 'language', 'credit', 'bought']

D  (maxDF=2500)  vs  B
    28 terms of B are gone. The most frequent of them:
       ['movie', 'film', 'one', 'like', 'good', 'even', 'time', 'really', 'see', 'story', 'much', 'well', 'get', 'people']
    28 terms take their place. The first of them:
       ['frank', 'language', 'credit', 'bought', 'disturbing', 'singing', 'taste', 'audiences', 'channel', 'surprisingly', 'army', 'focus', 'successful', 'concept']



In [6]:
# The claim behind maxDF, checked: every term C dropped from the top of B's list really does
# appear in more than 5000 documents.
tokenized = runs["B  vocabSize=1000"].transform(docs).select("filtered").cache()

for term in dropped[:6]:
    df_count = tokenized.filter(F.array_contains("filtered", term)).count()
    print(f"  {term:<10} appears in {df_count:>6} of {n_docs} documents"
          f"   {'> 2500, so D drops it' if df_count > 2500 else '?'}")

  movie      appears in   7477 of 12520 documents   > 2500, so D drops it
  film       appears in   6750 of 12520 documents   > 2500, so D drops it
  one        appears in   6922 of 12520 documents   > 2500, so D drops it
  like       appears in   5562 of 12520 documents   > 2500, so D drops it
  good       appears in   4651 of 12520 documents   > 2500, so D drops it
  even       appears in   4070 of 12520 documents   > 2500, so D drops it


In [7]:
# And the claim behind minDF: a term C admitted is in at least 10 documents, and a term
# neither vocabulary contains may be in far fewer.
for term in [t for t in d if t not in set(b)][:3] + ["zzyzx", "supercalifragilistic"]:
    df_count = tokenized.filter(F.array_contains("filtered", term)).count()
    print(f"  {term:<22} {df_count:>6} documents")

  frank                     173 documents
  language                  204 documents
  credit                    220 documents
  zzyzx                       0 documents
  supercalifragilistic        0 documents


---

## 4. A dimension index means nothing without its vocabulary

Dimension 7 is a different word in each of the three runs. A model trained on one of these feature
spaces and scored on another would not error — the vectors would be the right shape — it would
simply be reading the wrong coordinates.

In [8]:
print(f"{'dim':>4}  " + "".join(f"{name.split()[0]:<15}" for name in vocabs))
for i in range(10):
    row = "".join(f"{vocab[i]!r:<15}" for vocab in vocabs.values())
    print(f"{i:>4}  {row}")

 dim  A              B              C              D              
   0  'movie'        'movie'        'good'         'films'        
   1  'film'         'film'         'even'         'never'        
   2  'one'          'one'          'time'         'many'         
   3  'like'         'like'         'really'       'two'          
   4  'good'         'good'         'see'          'love'         
   5  'even'         'even'         'story'        'character'    
   6  'time'         'time'         'much'         'best'         
   7  'really'       'really'       'well'         'know'         
   8  'see'          'see'          'get'          'little'       
   9  'story'        'story'        'people'       'plot'         


In [9]:
for label, other in (("C", c), ("D", d)):
    overlap = len(set(b) & set(other))
    same_place = sum(1 for i in range(min(len(b), len(other))) if b[i] == other[i])
    print(f"B and {label} share {overlap} terms ({overlap / len(b):.1%} of B), "
          f"and put {same_place} of them at the same index.")

B and C share 996 terms (99.6% of B), and put 0 of them at the same index.
B and D share 972 terms (97.2% of B), and put 0 of them at the same index.


This is the concrete form of the chapter's warning about fitting a vectorizer twice. The failure
is not that the two vocabularies differ in size — that would be caught by the next stage. It is
that they differ in **order**, so the shapes agree and the meanings do not.

---

## 5. What the settings do to the representation

Two things change with the vocabulary: how many coordinates a document vector has, and how many
of them are non-zero.

In [10]:
from pyspark.ml.functions import vector_to_array

print(f"{'run':<42}{'dims':>7}{'mean non-zero':>15}{'density':>10}")
print("-" * 74)
stats = {}
for name, model in runs.items():
    arr = vector_to_array("rawFeatures")
    row = (model.transform(docs)
           .select(F.size(arr).alias("d"),
                   F.size(F.filter(arr, lambda x: x > 0)).alias("nz"))
           .agg(F.first("d").alias("dims"), F.avg("nz").alias("mean_nz"))
           .first())
    stats[name] = row
    print(f"{name:<42}{row['dims']:>7}{row['mean_nz']:>15.1f}"
          f"{row['mean_nz'] / row['dims']:>10.1%}")

run                                          dims  mean non-zero   density
--------------------------------------------------------------------------


A  vocabSize=500                              500           40.0      8.0%


B  vocabSize=1000                            1000           51.6      5.2%


C  vocabSize=1000, minDF=10, maxDF=5000      1000           49.5      5.0%


D  vocabSize=1000, minDF=10, maxDF=2500      1000           43.7      4.4%


Doubling the vocabulary from 500 to 1000 does **not** double the non-zeros per document. The
terms added by the second five hundred are rarer by construction, so most documents do not
contain them, and the vectors get longer faster than they get fuller. Density falls as the
vocabulary grows, which is the property that makes sparse storage worth having.

In [11]:
# The chapter's arithmetic for a realistic vocabulary, and this corpus measured against it.
def storage(dim, nnz, dense_bytes=8, sparse_bytes=12):
    return dim * dense_bytes, nnz * sparse_bytes


for label, dim, nnz in [("this corpus, run B", 1000, stats["B  vocabSize=1000"]["mean_nz"]),
                        ("the chapter's example", 100_000, 200)]:
    dense, sparse = storage(dim, nnz)
    print(f"{label:<24} {dim:>7} dims, {nnz:>6.0f} non-zero -> "
          f"dense {dense / 1024:8.1f} kB   sparse {sparse / 1024:6.1f} kB   "
          f"ratio {dense / sparse:5.0f}x")

this corpus, run B          1000 dims,     52 non-zero -> dense      7.8 kB   sparse    0.6 kB   ratio    13x
the chapter's example     100000 dims,    200 non-zero -> dense    781.2 kB   sparse    2.3 kB   ratio   333x


In [12]:
# The chapter quotes 800 kB, 2.4 kB and about 300x for the 10^5-term case.
dense, sparse = storage(100_000, 200)
assert round(dense / 1024) == 781 and round(sparse / 1024, 1) == 2.3
print(f"exactly: {dense} bytes dense, {sparse} bytes sparse, ratio {dense / sparse:.0f}")
print(f"the chapter rounds these to 800 kB, 2.4 kB and about 300x, "
      f"which is 1000 bytes to the kilobyte rather than 1024")

exactly: 800000 bytes dense, 2400 bytes sparse, ratio 333
the chapter rounds these to 800 kB, 2.4 kB and about 300x, which is 1000 bytes to the kilobyte rather than 1024


---

## 6. `minDF` as a fraction

`minDF` is read as a count when it is at least one and as a fraction of the corpus when it is
below one. The same is true of `maxDF`. This is easy to trip over: `minDF=1.0` is the default and
means *one document*, not *all of them*.

In [13]:
for value in (10.0, 0.02, 0.10, 0.30):
    v = fit_vectorizer(vocabSize=1000, minDF=value).stages[2].vocabulary
    reading = f"{value:.0f} documents" if value >= 1 else f"{value:.1%} of {n_docs} documents"
    print(f"minDF={value:<7} read as {reading:<28} -> {len(v):>5} terms")

minDF=10.0    read as 10 documents                 ->  1000 terms


minDF=0.02    read as 2.0% of 12520 documents      ->   842 terms


minDF=0.1     read as 10.0% of 12520 documents     ->   111 terms


minDF=0.3     read as 30.0% of 12520 documents     ->     8 terms


---

## What to take away

`vocabSize` is a ceiling on the feature space, not its size: the document-frequency thresholds are
applied first, and run C ended below its ceiling. `maxDF` removes the terms at the top of the
frequency list, doing automatically what a stop-word list does by name. `minDF` removes the tail,
where the terms too rare for a model to learn anything from live.

All three change **which term owns which dimension**, and none of them raises anything when they
do. That is the reason the vocabulary lives inside the fitted model rather than in the code that
built it.

Measured on this corpus of **12,520 documents**: a 500-term vocabulary leaves about 40 non-zeros
per document at 8.0 % density, and a 1,000-term vocabulary about 52 at 5.2 %. Doubling the
vocabulary did not double the non-zeros, because the terms it added are rarer — which is why
these vectors get sparser as the feature space grows.

In [14]:
spark.stop()